# Vesuvius Challenge - Surface Detection: EDA & Starter Notebook

**Competition:** [Vesuvius Challenge - Surface Detection](https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection)  
**Host:** Vesuvius Challenge / Scroll Prize  
**Prize Pool:** $100,000-$200,000  
**Deadline:** February 13, 2026  
**Type:** Research Code Competition  
**Author:** Lorenzo Scaturchio ([lorenzoscaturchio](https://www.kaggle.com/lorenzoscaturchio))

---

## Competition Overview

The Vesuvius Challenge asks us to **detect the papyrus surface inside 3D CT scans** of ancient scrolls from Herculaneum, buried by the eruption of Mount Vesuvius in 79 AD. These scrolls are too fragile to physically unroll, but micro-CT scanning reveals their internal structure.

### The Goal
Build a model that **segments the scroll's papyrus surface** in 3D CT volumes. This is the critical step in the virtual unwrapping pipeline that allows researchers to read the text on these ancient manuscripts.

### Key Facts
- **Data:** 3D CT scan chunks + binary surface masks
- **Task:** 3D semantic segmentation of papyrus surfaces
- **Metric:** Weighted blend of Surface Dice, TopoScore, and VOI (Variation of Information)
- **Why topology matters:** Clean, continuous surfaces are needed for the unwrapping pipeline -- gaps, holes, and sheet-switches break downstream processing
- **759 teams** = Large tier: Bronze top 10% (~76 teams)

## Medal Analysis

With **759 teams** this is a large-tier competition:

| Medal | Threshold | Approx. Position |
|-------|-----------|------------------|
| Bronze | Top 10% | Top ~76 |
| Silver | Top 5% | Top ~38 |
| Gold | Top 10 + 0.2% | Top ~10-15 |

**Strategy:** This is a classic computer vision segmentation problem with a unique twist (3D + topology-aware metrics). A strong 3D U-Net baseline with post-processing can compete for Bronze/Silver.

---
## Part 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q numpy pandas matplotlib seaborn scikit-image scipy torch torchvision monai nibabel tifffile albumentations

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
import seaborn as sns
from pathlib import Path
from scipy import ndimage
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Plotting config
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('darkgrid')
sns.set_palette('deep')

print('Environment ready.')
print(f'NumPy version: {np.__version__}')

## Part 2: Understanding the Data

### Data Structure
The competition provides **3D CT scan chunks** from ancient scrolls with corresponding **binary masks** indicating the papyrus surface locations.

```
vesuvius-challenge-surface-detection/
  train/
    chunk_001/
      volume.npy    # 3D CT volume (z, y, x)
      mask.npy      # Binary surface mask (z, y, x)
    chunk_002/
      ...
  test/
    chunk_xxx/
      volume.npy    # 3D CT volume only
  sample_submission.csv
```

Each chunk is a sub-volume from the full scroll CT scan. The mask labels indicate smoothed sheet positions (the papyrus surface).

In [ ]:
# Data paths (adjust for Kaggle kernel)
# On Kaggle: /kaggle/input/vesuvius-challenge-surface-detection/
DATA_DIR = Path('/kaggle/input/vesuvius-challenge-surface-detection')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'

# For local development, create synthetic data
LOCAL_MODE = not DATA_DIR.exists()
if LOCAL_MODE:
    print("Running in LOCAL MODE with synthetic data")
    print("On Kaggle, real data will be loaded from /kaggle/input/")
    TRAIN_DIR = Path('synthetic_train')
    TEST_DIR = Path('synthetic_test')
    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    TEST_DIR.mkdir(parents=True, exist_ok=True)
else:
    print(f"Data directory found: {DATA_DIR}")
    train_chunks = sorted(TRAIN_DIR.glob('*'))
    test_chunks = sorted(TEST_DIR.glob('*'))
    print(f"Training chunks: {len(train_chunks)}")
    print(f"Test chunks: {len(test_chunks)}")

In [ ]:
# Generate synthetic 3D CT data for local development / demonstration
def generate_synthetic_scroll_volume(shape=(64, 256, 256), n_layers=5, noise_level=0.15):
    """
    Generate synthetic 3D CT volume mimicking a rolled scroll cross-section.
    The scroll appears as concentric curved layers (papyrus sheets) in the CT.
    """
    np.random.seed(42)
    z, h, w = shape
    volume = np.random.normal(0.3, noise_level, shape).astype(np.float32)
    mask = np.zeros(shape, dtype=np.uint8)
    
    center_y, center_x = h // 2, w // 2
    for layer_idx in range(n_layers):
        radius = 30 + layer_idx * 25
        thickness = 2
        
        for z_slice in range(z):
            r_offset = np.sin(z_slice * 0.1) * 3
            for angle in np.linspace(0, 2 * np.pi, 500):
                r = radius + r_offset + np.sin(angle * 3) * 5
                y = int(center_y + r * np.sin(angle))
                x = int(center_x + r * np.cos(angle))
                
                if 0 <= y < h and 0 <= x < w:
                    for dy in range(-thickness, thickness+1):
                        for dx in range(-thickness, thickness+1):
                            ny, nx = y + dy, x + dx
                            if 0 <= ny < h and 0 <= nx < w:
                                volume[z_slice, ny, nx] = 0.8 + np.random.normal(0, 0.05)
                                if abs(dy) <= 1 and abs(dx) <= 1:
                                    mask[z_slice, ny, nx] = 1
    
    volume = np.clip(volume, 0, 1)
    return volume, mask


if LOCAL_MODE:
    print("Generating synthetic scroll data...")
    n_train_chunks = 4
    n_test_chunks = 2
    
    train_volumes = []
    train_masks = []
    
    for i in range(n_train_chunks):
        chunk_dir = TRAIN_DIR / f'chunk_{i:03d}'
        chunk_dir.mkdir(exist_ok=True)
        vol, msk = generate_synthetic_scroll_volume(
            shape=(64, 256, 256), 
            n_layers=3 + i,
            noise_level=0.1 + i * 0.02
        )
        np.save(chunk_dir / 'volume.npy', vol)
        np.save(chunk_dir / 'mask.npy', msk)
        train_volumes.append(vol)
        train_masks.append(msk)
        print(f"  Chunk {i}: volume {vol.shape}, mask {msk.shape}, "
              f"surface voxels: {msk.sum():,} ({msk.mean():.2%})")
    
    for i in range(n_test_chunks):
        chunk_dir = TEST_DIR / f'chunk_{100+i:03d}'
        chunk_dir.mkdir(exist_ok=True)
        vol, _ = generate_synthetic_scroll_volume(shape=(64, 256, 256))
        np.save(chunk_dir / 'volume.npy', vol)
    
    print(f"\nGenerated {n_train_chunks} training + {n_test_chunks} test chunks.")

## Part 3: Volume Exploration & Visualization

In [ ]:
# Load a training chunk for exploration
chunk_path = sorted(TRAIN_DIR.glob('chunk_*'))[0]
volume = np.load(chunk_path / 'volume.npy')
mask = np.load(chunk_path / 'mask.npy')

print(f"Volume shape: {volume.shape}")
print(f"Volume dtype: {volume.dtype}")
print(f"Volume range: [{volume.min():.4f}, {volume.max():.4f}]")
print(f"Volume mean: {volume.mean():.4f}, std: {volume.std():.4f}")
print(f"\nMask shape: {mask.shape}")
print(f"Mask dtype: {mask.dtype}")
print(f"Mask unique values: {np.unique(mask)}")
print(f"Surface voxels: {mask.sum():,} / {mask.size:,} ({mask.mean():.4%})")

In [ ]:
# Visualize CT slices with mask overlay
n_slices = 8
slice_indices = np.linspace(0, volume.shape[0]-1, n_slices, dtype=int)

fig, axes = plt.subplots(3, n_slices, figsize=(24, 10))

for col, z_idx in enumerate(slice_indices):
    axes[0, col].imshow(volume[z_idx], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'z={z_idx}', fontsize=10)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(mask[z_idx], cmap='Reds', vmin=0, vmax=1)
    axes[1, col].axis('off')
    
    axes[2, col].imshow(volume[z_idx], cmap='gray', vmin=0, vmax=1)
    mask_overlay = np.ma.masked_where(mask[z_idx] == 0, mask[z_idx])
    axes[2, col].imshow(mask_overlay, cmap='autumn', alpha=0.6, vmin=0, vmax=1)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('CT Volume', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Surface Mask', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Overlay', fontsize=12, fontweight='bold')

plt.suptitle('3D CT Scroll Volume - Axial Slices', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('volume_slices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Orthogonal views (axial, coronal, sagittal)
mid_z = volume.shape[0] // 2
mid_y = volume.shape[1] // 2
mid_x = volume.shape[2] // 2

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

views = [
    ('Axial (z)', volume[mid_z], mask[mid_z]),
    ('Coronal (y)', volume[:, mid_y, :], mask[:, mid_y, :]),
    ('Sagittal (x)', volume[:, :, mid_x], mask[:, :, mid_x]),
]

for col, (title, vol_slice, mask_slice) in enumerate(views):
    axes[0, col].imshow(vol_slice, cmap='gray', aspect='auto')
    axes[0, col].set_title(f'{title} - CT', fontweight='bold')
    axes[0, col].axis('off')
    
    axes[1, col].imshow(vol_slice, cmap='gray', aspect='auto')
    mask_overlay = np.ma.masked_where(mask_slice == 0, mask_slice)
    axes[1, col].imshow(mask_overlay, cmap='hot', alpha=0.7, aspect='auto')
    axes[1, col].set_title(f'{title} - Overlay', fontweight='bold')
    axes[1, col].axis('off')

plt.suptitle('Orthogonal Views of Scroll CT Volume', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('orthogonal_views.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Intensity distribution analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(volume.flatten(), bins=100, color='steelblue', alpha=0.7, density=True)
axes[0].set_title('Overall Voxel Intensity Distribution', fontweight='bold')
axes[0].set_xlabel('Intensity')
axes[0].set_ylabel('Density')
axes[0].axvline(volume.mean(), color='red', linestyle='--', label=f'Mean: {volume.mean():.3f}')
axes[0].legend()

surface_voxels = volume[mask == 1]
background_voxels = volume[mask == 0]
axes[1].hist(background_voxels, bins=100, alpha=0.6, color='gray', density=True, label='Background')
axes[1].hist(surface_voxels, bins=100, alpha=0.6, color='red', density=True, label='Surface')
axes[1].set_title('Surface vs Background Intensity', fontweight='bold')
axes[1].set_xlabel('Intensity')
axes[1].set_ylabel('Density')
axes[1].legend()

slice_ratios = [mask[z].mean() for z in range(mask.shape[0])]
axes[2].plot(range(len(slice_ratios)), slice_ratios, 'b-o', markersize=3)
axes[2].fill_between(range(len(slice_ratios)), slice_ratios, alpha=0.3)
axes[2].set_title('Surface Ratio per Z-Slice', fontweight='bold')
axes[2].set_xlabel('Z Slice')
axes[2].set_ylabel('Surface Voxel Ratio')
axes[2].set_ylim(0, max(slice_ratios) * 1.2)

plt.tight_layout()
plt.savefig('intensity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSurface voxel intensity: mean={surface_voxels.mean():.4f}, std={surface_voxels.std():.4f}")
print(f"Background intensity: mean={background_voxels.mean():.4f}, std={background_voxels.std():.4f}")
print(f"Separation: {abs(surface_voxels.mean() - background_voxels.mean()):.4f}")

## Part 4: Cross-Chunk Analysis

In [ ]:
# Analyze all training chunks
chunk_stats = []
for chunk_path in sorted(TRAIN_DIR.glob('chunk_*')):
    vol = np.load(chunk_path / 'volume.npy')
    msk = np.load(chunk_path / 'mask.npy')
    
    labeled, n_components = ndimage.label(msk)
    component_sizes = ndimage.sum(msk, labeled, range(1, n_components + 1))
    
    stats = {
        'chunk': chunk_path.name,
        'shape': str(vol.shape),
        'vol_mean': vol.mean(),
        'vol_std': vol.std(),
        'surface_ratio': msk.mean(),
        'surface_voxels': msk.sum(),
        'n_components': n_components,
        'largest_component': max(component_sizes) if len(component_sizes) > 0 else 0,
        'surface_mean_intensity': vol[msk == 1].mean() if msk.sum() > 0 else 0,
        'bg_mean_intensity': vol[msk == 0].mean(),
    }
    chunk_stats.append(stats)

stats_df = pd.DataFrame(chunk_stats)
print("Cross-Chunk Statistics")
print("=" * 80)
print(stats_df.to_string(index=False))

In [ ]:
# Visualize cross-chunk comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(stats_df['chunk'], stats_df['surface_ratio'], color='coral')
axes[0, 0].set_title('Surface Voxel Ratio per Chunk', fontweight='bold')
axes[0, 0].set_ylabel('Ratio')
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].bar(stats_df['chunk'], stats_df['n_components'], color='steelblue')
axes[0, 1].set_title('Connected Components per Chunk', fontweight='bold')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)

x = np.arange(len(stats_df))
width = 0.35
axes[1, 0].bar(x - width/2, stats_df['surface_mean_intensity'], width, 
               label='Surface', color='red', alpha=0.7)
axes[1, 0].bar(x + width/2, stats_df['bg_mean_intensity'], width, 
               label='Background', color='gray', alpha=0.7)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(stats_df['chunk'], rotation=45)
axes[1, 0].set_title('Mean Intensity: Surface vs Background', fontweight='bold')
axes[1, 0].legend()

axes[1, 1].bar(x - width/2, stats_df['vol_mean'], width, 
               label='Mean', color='green', alpha=0.7)
axes[1, 1].bar(x + width/2, stats_df['vol_std'], width, 
               label='Std', color='orange', alpha=0.7)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(stats_df['chunk'], rotation=45)
axes[1, 1].set_title('Volume Intensity Statistics', fontweight='bold')
axes[1, 1].legend()

plt.suptitle('Cross-Chunk Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('cross_chunk_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5: Understanding the Evaluation Metric

The competition uses a **topology-aware weighted blend** of three metrics:

1. **Surface Dice** -- Measures voxel-level accuracy of the predicted surface
2. **TopoScore** -- Penalizes topological errors (gaps, holes, sheet-switches, mergers)
3. **VOI (Variation of Information)** -- Information-theoretic measure of segmentation quality

The final score rewards both **voxel accuracy AND surface connectivity** -- you cannot win with accuracy alone.

In [ ]:
# Implement evaluation metrics

def surface_dice(pred, gt, tolerance=1):
    """
    Surface Dice coefficient.
    Measures overlap at the surface boundary with a distance tolerance.
    """
    from scipy.ndimage import distance_transform_edt
    
    pred_dist = distance_transform_edt(~pred.astype(bool))
    gt_dist = distance_transform_edt(~gt.astype(bool))
    
    pred_surface = pred.astype(bool) & (distance_transform_edt(pred.astype(bool)) <= 1)
    gt_surface = gt.astype(bool) & (distance_transform_edt(gt.astype(bool)) <= 1)
    
    pred_match = (gt_dist[pred_surface] <= tolerance).sum()
    gt_match = (pred_dist[gt_surface] <= tolerance).sum()
    
    total = pred_surface.sum() + gt_surface.sum()
    if total == 0:
        return 1.0
    
    return (pred_match + gt_match) / total


def variation_of_information(pred, gt):
    """
    Variation of Information (VOI) between two segmentations.
    Lower is better.
    """
    n = pred.size
    pred_flat = pred.flatten().astype(int)
    gt_flat = gt.flatten().astype(int)
    
    p11 = ((pred_flat == 1) & (gt_flat == 1)).sum() / n
    p10 = ((pred_flat == 1) & (gt_flat == 0)).sum() / n
    p01 = ((pred_flat == 0) & (gt_flat == 1)).sum() / n
    p00 = ((pred_flat == 0) & (gt_flat == 0)).sum() / n
    
    p_pred1 = p11 + p10
    p_pred0 = p01 + p00
    p_gt1 = p11 + p01
    p_gt0 = p10 + p00
    
    def safe_log(x):
        return np.log2(x) if x > 0 else 0
    
    h_pred_given_gt = 0
    for p_joint, p_marg in [(p11, p_gt1), (p10, p_gt0), (p01, p_gt1), (p00, p_gt0)]:
        if p_joint > 0 and p_marg > 0:
            h_pred_given_gt -= p_joint * safe_log(p_joint / p_marg)
    
    h_gt_given_pred = 0
    for p_joint, p_marg in [(p11, p_pred1), (p01, p_pred0), (p10, p_pred1), (p00, p_pred0)]:
        if p_joint > 0 and p_marg > 0:
            h_gt_given_pred -= p_joint * safe_log(p_joint / p_marg)
    
    return h_pred_given_gt + h_gt_given_pred


def compute_competition_score(pred, gt):
    """Approximate the competition's weighted metric blend."""
    sd = surface_dice(pred, gt)
    voi = variation_of_information(pred, gt)
    voi_score = max(0, 1 - voi)
    score = 0.4 * sd + 0.3 * voi_score + 0.3 * sd
    return {
        'surface_dice': sd,
        'voi': voi,
        'voi_score': voi_score,
        'combined_score': score
    }

print("Evaluation metrics defined.")
print("  Surface Dice: Boundary overlap [0,1], higher = better")
print("  VOI: Information distance [0,inf), lower = better")
print("  Combined: Weighted blend [0,1], higher = better")

In [ ]:
# Test metrics with synthetic predictions
gt_mask = mask.copy()

predictions = {
    'Perfect': gt_mask.copy(),
    'Eroded (1px)': ndimage.binary_erosion(gt_mask, iterations=1).astype(np.uint8),
    'Dilated (1px)': ndimage.binary_dilation(gt_mask, iterations=1).astype(np.uint8),
    'Noisy (5%)': (gt_mask ^ (np.random.random(gt_mask.shape) < 0.05)).astype(np.uint8),
    'Noisy (10%)': (gt_mask ^ (np.random.random(gt_mask.shape) < 0.10)).astype(np.uint8),
    'Empty': np.zeros_like(gt_mask),
}

z_mid = gt_mask.shape[0] // 2
gt_slice = gt_mask[z_mid]

scores = []
for name, pred in predictions.items():
    pred_slice = pred[z_mid]
    result = compute_competition_score(pred_slice, gt_slice)
    result['prediction'] = name
    scores.append(result)

scores_df = pd.DataFrame(scores)[['prediction', 'surface_dice', 'voi', 'combined_score']]
print("Metric Sensitivity Analysis")
print("=" * 60)
print(scores_df.to_string(index=False))

In [ ]:
# Visualize metric sensitivity
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

names = scores_df['prediction']
x = range(len(names))

axes[0].bar(x, scores_df['surface_dice'], color='#3498db')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].set_title('Surface Dice (higher = better)', fontweight='bold')
axes[0].set_ylim(0, 1.1)

axes[1].bar(x, scores_df['voi'], color='#e74c3c')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(names, rotation=45, ha='right')
axes[1].set_title('VOI (lower = better)', fontweight='bold')

axes[2].bar(x, scores_df['combined_score'], color='#2ecc71')
axes[2].set_xticks(list(x))
axes[2].set_xticklabels(names, rotation=45, ha='right')
axes[2].set_title('Combined Score (higher = better)', fontweight='bold')
axes[2].set_ylim(0, 1.1)

plt.suptitle('Evaluation Metric Sensitivity to Prediction Quality', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('metric_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 6: Baseline Model - 3D U-Net

Our baseline approach uses a **3D U-Net** architecture, the standard for volumetric medical image segmentation.

### Architecture Overview
- **Encoder:** 4 downsampling blocks with 3D convolutions
- **Bottleneck:** Deep feature representation
- **Decoder:** 4 upsampling blocks with skip connections
- **Output:** Binary surface probability map

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock3D(nn.Module):
    """Double 3D convolution block with BatchNorm and ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.block(x)


class UNet3D(nn.Module):
    """
    3D U-Net for volumetric segmentation of papyrus surfaces in CT scans.
    """
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.upconvs = nn.ModuleList()
        
        prev_ch = in_channels
        for feat in features:
            self.encoders.append(ConvBlock3D(prev_ch, feat))
            prev_ch = feat
        
        self.bottleneck = ConvBlock3D(features[-1], features[-1] * 2)
        
        for feat in reversed(features):
            self.upconvs.append(
                nn.ConvTranspose3d(feat * 2, feat, kernel_size=2, stride=2)
            )
            self.decoders.append(ConvBlock3D(feat * 2, feat))
        
        self.output = nn.Conv3d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x):
        skip_connections = []
        for encoder in self.encoders:
            x = encoder(x)
            skip_connections.append(x)
            x = self.pool(x)
        
        x = self.bottleneck(x)
        
        skip_connections = skip_connections[::-1]
        for idx in range(len(self.decoders)):
            x = self.upconvs[idx](x)
            skip = skip_connections[idx]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([skip, x], dim=1)
            x = self.decoders[idx](x)
        
        return self.output(x)


model = UNet3D(in_channels=1, out_channels=1, features=[32, 64, 128, 256])
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"3D U-Net Architecture")
print(f"=" * 50)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.1f} MB (float32)")

dummy_input = torch.randn(1, 1, 32, 64, 64)
dummy_output = model(dummy_input)
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}")

In [ ]:
# Dataset and DataLoader
from torch.utils.data import Dataset, DataLoader


class ScrollDataset(Dataset):
    """Dataset for scroll surface detection with random sub-volume cropping."""
    def __init__(self, data_dir, crop_size=(32, 128, 128), augment=False):
        self.data_dir = Path(data_dir)
        self.chunks = sorted(self.data_dir.glob('chunk_*'))
        self.crop_size = crop_size
        self.augment = augment
        
        self.volumes = []
        self.masks = []
        for chunk in self.chunks:
            self.volumes.append(np.load(chunk / 'volume.npy'))
            mask_path = chunk / 'mask.npy'
            if mask_path.exists():
                self.masks.append(np.load(mask_path))
            else:
                self.masks.append(None)
        print(f"Loaded {len(self.chunks)} chunks")
    
    def __len__(self):
        return len(self.chunks) * 10
    
    def __getitem__(self, idx):
        chunk_idx = idx % len(self.chunks)
        vol = self.volumes[chunk_idx]
        msk = self.masks[chunk_idx]
        
        dz, dy, dx = self.crop_size
        z0 = np.random.randint(0, max(1, vol.shape[0] - dz))
        y0 = np.random.randint(0, max(1, vol.shape[1] - dy))
        x0 = np.random.randint(0, max(1, vol.shape[2] - dx))
        
        vol_crop = vol[z0:z0+dz, y0:y0+dy, x0:x0+dx].copy()
        
        if self.augment:
            if np.random.random() > 0.5:
                vol_crop = np.flip(vol_crop, axis=1).copy()
            if np.random.random() > 0.5:
                vol_crop = np.flip(vol_crop, axis=2).copy()
            vol_crop = vol_crop * np.random.uniform(0.9, 1.1)
            vol_crop = np.clip(vol_crop, 0, 1)
        
        vol_tensor = torch.from_numpy(vol_crop).float().unsqueeze(0)
        
        if msk is not None:
            msk_crop = msk[z0:z0+dz, y0:y0+dy, x0:x0+dx].copy()
            if self.augment:
                if np.random.random() > 0.5:
                    msk_crop = np.flip(msk_crop, axis=1).copy()
                if np.random.random() > 0.5:
                    msk_crop = np.flip(msk_crop, axis=2).copy()
            msk_tensor = torch.from_numpy(msk_crop).float().unsqueeze(0)
            return vol_tensor, msk_tensor
        
        return vol_tensor


train_dataset = ScrollDataset(TRAIN_DIR, crop_size=(32, 128, 128), augment=True)
print(f"Dataset length: {len(train_dataset)}")

sample_vol, sample_mask = train_dataset[0]
print(f"Sample volume: {sample_vol.shape}")
print(f"Sample mask: {sample_mask.shape}, surface ratio: {sample_mask.mean():.4f}")

## Part 7: Training Loop

In [ ]:
# Loss functions

class DiceBCELoss(nn.Module):
    """Combined Dice + BCE loss for surface detection."""
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        pred_sig = torch.sigmoid(pred)
        intersection = (pred_sig * target).sum()
        dice = (2. * intersection + self.smooth) / (pred_sig.sum() + target.sum() + self.smooth)
        dice_loss = 1 - dice
        return self.dice_weight * dice_loss + self.bce_weight * bce_loss


class TopologyAwareLoss(nn.Module):
    """Topology-aware loss combining Dice+BCE with connectivity penalty."""
    def __init__(self, base_weight=0.8, topo_weight=0.2):
        super().__init__()
        self.base_loss = DiceBCELoss()
        self.base_weight = base_weight
        self.topo_weight = topo_weight
    
    def forward(self, pred, target):
        base = self.base_loss(pred, target)
        pred_sig = torch.sigmoid(pred)
        laplacian = (
            torch.abs(pred_sig[:,:,1:,:,:] - pred_sig[:,:,:-1,:,:]).mean() +
            torch.abs(pred_sig[:,:,:,1:,:] - pred_sig[:,:,:,:-1,:]).mean() +
            torch.abs(pred_sig[:,:,:,:,1:] - pred_sig[:,:,:,:,:-1]).mean()
        )
        return self.base_weight * base + self.topo_weight * laplacian

print("Loss functions defined.")

In [ ]:
# Training loop
def train_model(model, train_loader, epochs=5, lr=1e-3, device='cpu'):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = TopologyAwareLoss()
    
    history = {'loss': [], 'dice': [], 'lr': []}
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        epoch_dice = 0
        n_batches = 0
        
        for batch_idx, (volumes, masks) in enumerate(train_loader):
            volumes = volumes.to(device)
            masks = masks.to(device)
            
            outputs = model(volumes)
            loss = criterion(outputs, masks)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            with torch.no_grad():
                pred = (torch.sigmoid(outputs) > 0.5).float()
                intersection = (pred * masks).sum()
                dice = (2 * intersection) / (pred.sum() + masks.sum() + 1e-8)
            
            epoch_loss += loss.item()
            epoch_dice += dice.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / max(n_batches, 1)
        avg_dice = epoch_dice / max(n_batches, 1)
        current_lr = scheduler.get_last_lr()[0]
        
        history['loss'].append(avg_loss)
        history['dice'].append(avg_dice)
        history['lr'].append(current_lr)
        
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, Dice: {avg_dice:.4f}, LR: {current_lr:.6f}")
    
    return model, history


train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training device: {device}")

model = UNet3D(in_channels=1, out_channels=1, features=[16, 32, 64, 128])
model, history = train_model(model, train_loader, epochs=5, lr=1e-3, device=device)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['loss'], 'b-o', markersize=6)
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(history['dice'], 'g-o', markersize=6)
axes[1].set_title('Dice Score', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].set_ylim(0, 1)

axes[2].plot(history['lr'], 'r-o', markersize=6)
axes[2].set_title('Learning Rate Schedule', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')

plt.suptitle('3D U-Net Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 8: Inference & Post-Processing

In [ ]:
# Inference with sliding window
def predict_volume(model, volume, crop_size=(32, 128, 128), stride=(16, 64, 64), 
                   device='cpu', threshold=0.5):
    """Predict on a full volume using overlapping sliding windows."""
    model.eval()
    D, H, W = volume.shape
    dz, dy, dx = crop_size
    sz, sy, sx = stride
    
    pred_sum = np.zeros_like(volume, dtype=np.float32)
    count = np.zeros_like(volume, dtype=np.float32)
    
    with torch.no_grad():
        for z0 in range(0, max(1, D - dz + 1), sz):
            for y0 in range(0, max(1, H - dy + 1), sy):
                for x0 in range(0, max(1, W - dx + 1), sx):
                    crop = volume[z0:z0+dz, y0:y0+dy, x0:x0+dx]
                    if crop.shape != (dz, dy, dx):
                        continue
                    input_tensor = torch.from_numpy(crop).float().unsqueeze(0).unsqueeze(0).to(device)
                    output = torch.sigmoid(model(input_tensor))
                    pred_crop = output.cpu().numpy()[0, 0]
                    pred_sum[z0:z0+dz, y0:y0+dy, x0:x0+dx] += pred_crop
                    count[z0:z0+dz, y0:y0+dy, x0:x0+dx] += 1
    
    pred_avg = np.divide(pred_sum, count, where=count > 0, out=np.zeros_like(pred_sum))
    pred_binary = (pred_avg > threshold).astype(np.uint8)
    return pred_avg, pred_binary


# Post-processing
def postprocess_prediction(pred_binary, min_component_size=100):
    """Topology-aware post-processing: remove small components, fill holes, smooth."""
    from scipy.ndimage import label, binary_fill_holes, binary_closing, binary_opening
    
    labeled, n_components = label(pred_binary)
    print(f"  Raw components: {n_components}")
    
    cleaned = np.zeros_like(pred_binary)
    for i in range(1, n_components + 1):
        component = (labeled == i)
        if component.sum() >= min_component_size:
            cleaned[component] = 1
    
    for z in range(cleaned.shape[0]):
        cleaned[z] = binary_fill_holes(cleaned[z]).astype(np.uint8)
    
    struct = ndimage.generate_binary_structure(3, 1)
    cleaned = binary_closing(cleaned, structure=struct, iterations=1).astype(np.uint8)
    cleaned = binary_opening(cleaned, structure=struct, iterations=1).astype(np.uint8)
    
    labeled_clean, n_clean = label(cleaned)
    print(f"  Cleaned components: {n_clean}")
    return cleaned


# Run inference
test_vol = train_volumes[0] if LOCAL_MODE else np.load(sorted(TRAIN_DIR.glob('chunk_*'))[0] / 'volume.npy')
print(f"Running inference on volume: {test_vol.shape}")
pred_prob, pred_binary = predict_volume(model, test_vol, device=device)
print(f"Predicted surface voxels: {pred_binary.sum():,}")

print("\nPost-processing...")
pred_cleaned = postprocess_prediction(pred_binary, min_component_size=50)

In [ ]:
# Visualize predictions vs ground truth
gt_mask_viz = train_masks[0] if LOCAL_MODE else np.load(sorted(TRAIN_DIR.glob('chunk_*'))[0] / 'mask.npy')

z_mid = test_vol.shape[0] // 2
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

axes[0, 0].imshow(test_vol[z_mid], cmap='gray')
axes[0, 0].set_title('CT Volume')
axes[0, 1].imshow(gt_mask_viz[z_mid], cmap='Reds')
axes[0, 1].set_title('Ground Truth')
axes[0, 2].imshow(pred_prob[z_mid], cmap='hot', vmin=0, vmax=1)
axes[0, 2].set_title('Prediction (probability)')
axes[0, 3].imshow(pred_cleaned[z_mid], cmap='Greens')
axes[0, 3].set_title('Prediction (cleaned)')

for col, (title, pred) in enumerate([
    ('GT Overlay', gt_mask_viz[z_mid]),
    ('Raw Pred Overlay', pred_binary[z_mid]),
    ('Cleaned Pred Overlay', pred_cleaned[z_mid]),
]):
    axes[1, col].imshow(test_vol[z_mid], cmap='gray')
    overlay = np.ma.masked_where(pred == 0, pred)
    axes[1, col].imshow(overlay, cmap='autumn', alpha=0.6)
    axes[1, col].set_title(title)

diff = pred_cleaned[z_mid].astype(int) - gt_mask_viz[z_mid].astype(int)
axes[1, 3].imshow(diff, cmap='RdBu', vmin=-1, vmax=1)
axes[1, 3].set_title('Difference (Red=FP, Blue=FN)')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Prediction Results', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 9: Submission Generation

In [ ]:
def rle_encode(mask):
    """Run-length encoding for binary mask."""
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def generate_submission(model, test_dir, output_path='submission.csv', device='cpu'):
    """Generate competition submission file."""
    test_chunks = sorted(Path(test_dir).glob('chunk_*'))
    submissions = []
    
    for chunk_path in test_chunks:
        vol = np.load(chunk_path / 'volume.npy')
        print(f"Processing {chunk_path.name}: {vol.shape}")
        
        _, pred = predict_volume(model, vol, device=device)
        pred = postprocess_prediction(pred)
        
        rle = rle_encode(pred)
        submissions.append({'id': chunk_path.name, 'rle_mask': rle})
    
    sub_df = pd.DataFrame(submissions)
    sub_df.to_csv(output_path, index=False)
    print(f"\nSubmission saved to {output_path}")
    return sub_df


print("Generating submission...")
submission = generate_submission(model, TEST_DIR, device=device)
submission.head()

---
## Part 10: Strategic Insights & Medal Path

### Competition Strategy (4-Phase Plan)

| Phase | Timeline | Goal | Target |
|-------|----------|------|--------|
| 1. Baseline | Week 1-2 | 3D U-Net + DiceBCE | Establish score |
| 2. Architecture | Week 3-4 | nnU-Net / Swin UNETR | Top 50% |
| 3. Topology | Week 5-6 | Topology-aware losses + post-processing | Top 10% (Bronze) |
| 4. Polish | Final week | Ensemble + TTA + hyperparameter sweep | Silver/Gold push |

### Key Differentiators for Medal
- **Topology-aware loss functions** (clDice, persistent homology)
- **Multi-scale ensemble** (different crop sizes for local + global context)
- **Post-processing pipeline** tuned to the 3 competition metrics
- **Surface mesh refinement** after initial voxel prediction

### Why This Competition is Winnable
1. Well-studied domain (3D medical segmentation) -- leverage existing architectures
2. Topology-aware metric is the differentiator -- most teams optimize Dice alone
3. Prior Vesuvius Challenge (Ink Detection) has public gold solutions to learn from
4. Research format prevents leaderboard manipulation

In [ ]:
# Final summary
print("\n" + "="*60)
print("Vesuvius Challenge - Surface Detection - Summary")
print("="*60)
print(f"Competition: Vesuvius Challenge - Surface Detection")
print(f"Prize Pool: $100K-$200K")
print(f"Deadline: February 13, 2026")
print(f"Teams: ~759 (Large tier)")
print(f"Medal Thresholds: Bronze ~76th, Silver ~38th, Gold ~10-15th")
print(f"\nBaseline: 3D U-Net with topology-aware loss")
print(f"Metric: Weighted blend of Surface Dice + TopoScore + VOI")
print(f"Key Insight: Topology quality matters as much as voxel accuracy")
print(f"\nAuthor: Lorenzo Scaturchio")
print(f"Profile: kaggle.com/lorenzoscaturchio")
print("="*60)